In [1]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [2]:
!pip install -q langchain-community chromadb faiss-cpu tiktoken pypdf langchain_huggingface python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/

### Input:

```
city="Bhubaneswar",
profile="Student",
activity="Outdoor walking",
duration_minutes=30
```


### Output
```
City: Bhubaneswar
PM2.5: ...
PM10: ...
Weather: ...
AQI Category: ...
Dominant Pollutant: ...
Recommendation: ...
Reason: ...
```

## Fetch Lat & Lon from city name

In [3]:
import requests
import pandas as pd

In [4]:
def get_coordinates(city,country_code="IN"):
  url="https://geocoding-api.open-meteo.com/v1/search"
  params= {
      "name": city,
      "count": 10,
      "language": "en",
      "format": "json",
      "countryCode": country_code
  }
  response=requests.get(url,params=params)
  data=response.json()

  if 'results' not in data:
    return None

  data=data['results'][0]

  return {
      "city": data['name'],
      "country": data['country'],
      "latitude": data['latitude'],
      "longitude": data['longitude']
  }

In [5]:
location=get_coordinates("Bhubaneswar","IN")
location

{'city': 'Bhubaneswar',
 'country': 'India',
 'latitude': 20.27241,
 'longitude': 85.83385}

## Fetch air quality for the lat and lon

In [6]:
def get_air_quality(lat,lon):
  url="https://air-quality-api.open-meteo.com/v1/air-quality"

  params= {
        "latitude": lat,
        "longitude": lon,
        "current": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi,european_aqi",
        "hourly": "pm10,pm2_5,us_aqi,european_aqi",
        "timezone": "auto"
    }

  response=requests.get(url,params=params)
  data=response.json()
  return data["current"]

In [7]:
pollution=get_air_quality(20.27241,85.83385)
pollution

{'time': '2026-06-14T23:30',
 'interval': 3600,
 'pm10': 86.1,
 'pm2_5': 75.9,
 'carbon_monoxide': 731.0,
 'nitrogen_dioxide': 38.8,
 'sulphur_dioxide': 13.5,
 'ozone': 53.0,
 'us_aqi': 113,
 'european_aqi': 73}

## Fetch weather data

In [8]:
def get_weather(lat,lon):
  url="https://api.open-meteo.com/v1/forecast"

  params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,weather_code",
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
  response=requests.get(url,params=params)
  data=response.json()
  return data["current"]

In [9]:
weather=get_weather(20.27241,85.83385)
weather

{'time': '2026-06-15T00:00',
 'interval': 900,
 'temperature_2m': 28.0,
 'relative_humidity_2m': 89,
 'wind_speed_10m': 4.2,
 'precipitation': 0.0,
 'weather_code': 95}

## Fetch the needfull among the 3 data

In [10]:
def extract_curr_data(location,pollution,weather):
  return {
        "city": location["city"],
        "country": location["country"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "time": pollution["time"],
        "pm2_5": pollution["pm2_5"],
        "pm10": pollution["pm10"],
        "us_aqi": pollution["us_aqi"],
        "european_aqi": pollution["european_aqi"],
        "carbon_monoxide": pollution["carbon_monoxide"],
        "nitrogen_dioxide": pollution["nitrogen_dioxide"],
        "sulphur_dioxide": pollution["sulphur_dioxide"],
        "ozone": pollution["ozone"],
        "temperature": weather["temperature_2m"],
        "humidity": weather["relative_humidity_2m"],
        "wind_speed": weather["wind_speed_10m"],
        "precipitation": weather["precipitation"]
    }

In [11]:
data=extract_curr_data(location,pollution,weather)
data

{'city': 'Bhubaneswar',
 'country': 'India',
 'latitude': 20.27241,
 'longitude': 85.83385,
 'time': '2026-06-14T23:30',
 'pm2_5': 75.9,
 'pm10': 86.1,
 'us_aqi': 113,
 'european_aqi': 73,
 'carbon_monoxide': 731.0,
 'nitrogen_dioxide': 38.8,
 'sulphur_dioxide': 13.5,
 'ozone': 53.0,
 'temperature': 28.0,
 'humidity': 89,
 'wind_speed': 4.2,
 'precipitation': 0.0}

## Basic AQI Rules

In [66]:
def get_pm25_category(pm25):
    if pm25 is None:
        return "Unknown"
    if pm25 <= 30:
        return "Good"
    elif pm25 <= 60:
        return "Satisfactory"
    elif pm25 <= 90:
        return "Moderately Polluted"
    elif pm25 <= 120:
        return "Poor"
    elif pm25 <= 250:
        return "Very Poor"
    else:
        return "Severe"


def get_pm10_category(pm10):
    if pm10 is None:
        return "Unknown"
    if pm10 <= 50:
        return "Good"
    elif pm10 <= 100:
        return "Satisfactory"
    elif pm10 <= 250:
        return "Moderately Polluted"
    elif pm10 <= 350:
        return "Poor"
    elif pm10 <= 430:
        return "Very Poor"
    else:
        return "Severe"

In [67]:
RANK = {
    "Unknown": 0,
    "Good": 1,
    "Satisfactory": 2,
    "Moderately Polluted": 3,
    "Poor": 4,
    "Very Poor": 5,
    "Severe": 6
}

In [68]:
def get_overall_category(pm25, pm10):
    pm25_category=get_pm25_category(pm25)
    pm10_category=get_pm10_category(pm10)

    if RANK[pm25_category]>=RANK[pm10_category]:
        return {
            "overall_category": pm25_category,
            "dominant_pollutant": "PM2.5",
            "pm25_category": pm25_category,
            "pm10_category": pm10_category
        }
    else:
        return {
            "overall_category": pm10_category,
            "dominant_pollutant": "PM10",
            "pm25_category": pm25_category,
            "pm10_category": pm10_category
        }

In [69]:
def generate_basic_recommendation(category,profile,activity):
    category=category.lower()
    activity=activity.lower()
    profile=profile.lower()

    if category in ["good","satisfactory"]:
        return "Outdoor activity is generally okay. Prefer low-traffic areas if possible."

    elif category=="moderately polluted":
        if "jog" in activity or "run" in activity or "cycling" in activity:
            return "You can go outside, but avoid long or intense outdoor exercise."
        else:
            return "Short outdoor activity is acceptable, but avoid staying outside for too long."

    elif category=="poor":
        if "jog" in activity or "run" in activity or "cycling" in activity:
            return "Avoid outdoor high-intensity exercise right now. Prefer indoor activity."
        else:
            return "Limit outdoor exposure and avoid polluted roads or traffic-heavy areas."

    elif category=="very poor":
        return "Avoid unnecessary outdoor activity. Keep outdoor exposure short."

    elif category=="severe":
        return "Avoid outdoor activity unless necessary."

    else:
        return "Air quality data is unavailable, so a reliable recommendation cannot be generated."

In [70]:
def get_air_quality_plan(city,profile,activity,duration_minutes):
    location=get_coordinates(city)

    if location is None:
        return {
            "error": "City not found. Please try another city name."
        }

    pollution=get_air_quality(location["latitude"], location["longitude"])
    weather=get_weather(location["latitude"], location["longitude"])

    data=extract_curr_data(location,pollution,weather)

    result=get_overall_category(data["pm2_5"],data["pm10"])

    suggestion=generate_basic_recommendation(result["overall_category"],profile,activity)

    final_output = {
        "city": data["city"],
        "country": data["country"],
        "time": data["time"],
        "pm2_5": data["pm2_5"],
        "pm10": data["pm10"],
        "us_aqi": data["us_aqi"],
        "european_aqi": data["european_aqi"],
        "temperature": data["temperature"],
        "humidity": data["humidity"],
        "wind_speed": data["wind_speed"],
        "precipitation": data["precipitation"],
        "pm25_category": result["pm25_category"],
        "pm10_category": result["pm10_category"],
        "overall_category": result["overall_category"],
        "dominant_pollutant": result["dominant_pollutant"],
        "profile": profile,
        "activity": activity,
        "duration_minutes": duration_minutes,
        "recommendation": suggestion
    }

    return final_output

## Step-1(a): Indexing(Document Ingestion)

In [12]:
from google.colab import files

uploaded = files.upload()

Saving CPCB_AQI.pdf to CPCB_AQI.pdf
Saving WHO_AQG.pdf to WHO_AQG.pdf


In [16]:
import logging

logging.getLogger("pypdf").setLevel(logging.ERROR)

In [17]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file=["CPCB_AQI.pdf","WHO_AQG.pdf"]

docs=[]
for pdf in pdf_file:
  loader=PyPDFLoader(pdf)
  pages=loader.load()
  docs.extend(pages)

print("Total pages: ",len(docs))

Total pages:  362


In [26]:
documents=[]

for doc in docs:
    text=doc.page_content.strip()

    if len(text)>50:
        documents.append(doc)

print("Original pages:", len(docs))
print("Clean pages:", len(documents))

Original pages: 362
Clean pages: 350


In [27]:
for doc in documents:
    source=doc.metadata.get("source","unknown")

    if "cpcb" in source.lower() or "aqi" in source.lower():
        doc.metadata["organization"]="CPCB"
    elif "who" in source.lower():
        doc.metadata["organization"]="WHO"
    else:
        doc.metadata["organization"]="Unknown"

    doc.metadata["project"]="Air Quality Action Planner"

In [29]:
documents[100].metadata

{'producer': 'macOS Version 11.6 (Build 20G165) Quartz PDFContext',
 'creator': 'Adobe InDesign 16.0 (Macintosh)',
 'creationdate': "D:20211117081120Z00'00'",
 'moddate': "D:20211117081120Z00'00'",
 'source': 'WHO_AQG.pdf',
 'total_pages': 300,
 'page': 42,
 'page_label': '43',
 'organization': 'WHO',
 'project': 'Air Quality Action Planner'}

## Step-1(b): Indexing(Text Splitting)

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks=text_splitter.split_documents(documents)
print(len(chunks))

7699


In [33]:
keywords = ["PM2.5", "PM10", "Good", "Poor", "Severe", "AQI", "health", "public"]

for keyword in keywords:
    count = sum(1 for chunk in chunks if keyword.lower() in chunk.page_content.lower())
    print(f"{keyword}: {count} chunks")

PM2.5: 23 chunks
PM10: 27 chunks
Good: 30 chunks
Poor: 25 chunks
Severe: 24 chunks
AQI: 90 chunks
health: 62 chunks
public: 27 chunks


In [34]:
chunks[10]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-06-14T18:44:09+00:00', 'source': 'CPCB_AQI.pdf', 'total_pages': 62, 'page': 3, 'page_label': '4', 'organization': 'CPCB', 'project': 'Air Quality Action Planner'}, page_content='as local emission surveys. \nBriefly, an AQI is useful for: (i) general public to know air quality in a simplified way, (ii) a politician \nto invoke quick actions, (iii) a deci sion maker to know the trend of ev ents and to chalk out corrective')

## Step-1(c & d): Indexing(Embeddings & Vector Store)

In [36]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding = HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    )
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embedding,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
vector_store.save_local("/content/drive/MyDrive/Colab Notebooks/LangChain/AQI Planner RAG/air_quality_faiss_index")

## Step-2: Retriever

In [47]:
# convert vector store into a retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

In [48]:
retrieved_docs = similarity_retriever.invoke("Broad guidelines for public during severe AQI")

In [49]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Source Chunk {i+1} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Organization:", doc.metadata.get("organization"))
    print(doc.page_content[:700])


--- Source Chunk 1 ---
Source: CPCB_AQI.pdf
Page: 58
Organization: CPCB
Broad guidelines for Public 
 
AQI is an initiative intended to enhance public awareness and involvement in efforts to 
improve air quality. People can contribute by maintaining vehicles properly (e.g. get PUC 
checks, replace car air filter,maintainright tire s pressure), following lane discipline & speed 
limits, avoiding prolong idling and turning off e ngines at red traffic signals. In addition to 
above, during severe or very poor AQI, people should minimize travel; avoid using private 
vehicles and instead use public transport, bi kes or walk, and carpool; use smaller vehicles 
(e.g. avoid SUVs).

--- Source Chunk 2 ---
Source: CPCB_AQI.pdf
Page: 6
Organization: CPCB
academicians, medical fraternity, research inst itutes, MoEF, advocacy groups, SPCBs and CPCB. 
The committee deliberated, disc ussed and devised consensus on the AQI system. The committee 
oversaw the progress of the projec t on a continual bas

## Step-3: Augmentation

In [123]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    temperature=0.2
)

llm = ChatHuggingFace(llm=llm)


# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from langchain_huggingface import HuggingFacePipeline
# import torch

# model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(model_id)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     dtype=torch.float16
# )

# pipe = pipeline(
#     task="text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,
#     temperature=0.2,
#     do_sample=True,
#     return_full_text=False
# )

# llm = HuggingFacePipeline(pipeline=pipe)

In [124]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
      You are an Air Quality Action Planner for India.

Answer the user's question using ONLY the provided CPCB/WHO guideline context.

Rules:
1. Use only the provided context.
2. Do not invent facts.
3. Do not give medical diagnosis.
4. Give practical public guidance.
5. If the context does not contain the answer, say:
   "I could not find this information in the provided guideline documents."
6. Mention the source organization when possible, such as CPCB or WHO.
7. Keep the answer simple and useful.
8. For Indian AQI categories, use only CPCB National AQI / IND-AQI categories.
9. Do not include categories from other countries such as USEPA, China, or EU unless the user specifically asks for comparison.
10. Do not invent formulas.
11. Do not use city ranking, healthy score, or AQIR formulas unless the user asks about city ranking.
12. Do not give medical diagnosis or medicine advice.

Guideline Context:
{context}

User Question:
{question}

    """,
    input_variables=['context','question']
)

In [125]:
def format_docs(docs):
    formatted_context = ""

    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown source")
        page = doc.metadata.get("page", "Unknown page")
        organization = doc.metadata.get("organization", "Unknown organization")

        formatted_context+=f"\n\n--- Source {i+1} ---\n"
        formatted_context+=f"Organization: {organization}\n"
        formatted_context+=f"Source file: {source}\n"
        formatted_context+=f"Page: {page}\n"
        formatted_context+=f"Content:\n{doc.page_content}\n"

    return formatted_context

In [126]:
def format_sources(docs):
    sources = []

    for i, doc in enumerate(docs):
        sources.append({
            "source_id": i + 1,
            "organization": doc.metadata.get("organization", "Unknown"),
            "source_file": doc.metadata.get("source", "Unknown"),
            "page": doc.metadata.get("page", "Unknown"),
            "preview": doc.page_content[:300]
        })

    return sources

## Building a Chain

In [127]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [128]:
parallel_chain=RunnableParallel({
    "context":retriever|RunnableLambda(format_docs),
    "question":RunnablePassthrough()
})

In [129]:
parser=StrOutputParser()

In [130]:
main_chain=parallel_chain|prompt|llm|parser

In [131]:
question="What should people do during very poor or severe AQI?"
main_chain.invoke(question)

'During very poor or severe AQI, people should minimize travel and avoid using private vehicles. Instead, they should use public transport, bicycles, or walk. Carpooling is also recommended. Additionally, they should use smaller vehicles to reduce emissions.'

In [132]:
docs = retriever.invoke(question)
sources = format_sources(docs)
sources

[{'source_id': 1,
  'organization': 'CPCB',
  'source_file': 'CPCB_AQI.pdf',
  'page': 58,
  'preview': 'Broad guidelines for Public \n \nAQI is an initiative intended to enhance public awareness and involvement in efforts to \nimprove air quality. People can contribute by maintaining vehicles properly (e.g. get PUC \nchecks, replace car air filter,maintainright tire s pressure), following lane discipline '},
 {'source_id': 2,
  'organization': 'CPCB',
  'source_file': 'CPCB_AQI.pdf',
  'page': 34,
  'preview': 'Poor 800 Unhealthy 795 Moderately \nPolluted 800 High 500 \nVery Poor 1600 Very \nUnhealthy 1580 Heavily \nPolluted 1600 Very \nhigh 500+ \nSevere 1600+ Hazardous    1580+ Severely \nPolluted 2620 \xa0\xa0 \xa0\xa0\n(a) USEPA (2013) (b)Gao (2013) (c) CAQI (2012)*converted from ppb to µg/m3 and rounded off'},
 {'source_id': 3,
  'organization': 'CPCB',
  'source_file': 'CPCB_AQI.pdf',
  'page': 48,
  'preview': 'used in arrivi\nAQI-based S\nprocedure, c\narameters ar\nessitate t

## Connecting AQI logic with RAG Pipeline

In [133]:
plan=get_air_quality_plan(
    city="Bhubaneswar",
    profile="Student",
    activity="Outdoor jogging",
    duration_minutes=30
)

In [134]:
plan

{'city': 'Bhubaneswar',
 'country': 'India',
 'time': '2026-06-15T01:30',
 'pm2_5': 88.7,
 'pm10': 97.8,
 'us_aqi': 117,
 'european_aqi': 74,
 'temperature': 27.6,
 'humidity': 92,
 'wind_speed': 4.8,
 'precipitation': 0.0,
 'pm25_category': 'Moderately Polluted',
 'pm10_category': 'Satisfactory',
 'overall_category': 'Moderately Polluted',
 'dominant_pollutant': 'PM2.5',
 'profile': 'Student',
 'activity': 'Outdoor jogging',
 'duration_minutes': 30,
 'recommendation': 'You can go outside, but avoid long or intense outdoor exercise.'}

In [145]:
rag_question = f"""
The user is asking for an air quality action recommendation.

Current air quality summary:
- Overall AQI category: {plan.get('overall_category')}
- Dominant pollutant: {plan.get('dominant_pollutant')}
- PM2.5: {plan.get('pm2_5')} µg/m³
- PM10: {plan.get('pm10')} µg/m³
- Carbon monoxide: {plan.get('carbon_monoxide')}
- Nitrogen dioxide: {plan.get('nitrogen_dioxide')}
- Sulphur dioxide: {plan.get('sulphur_dioxide')}
- Ozone: {plan.get('ozone')}
- US AQI: {plan.get('us_aqi')}
- European AQI: {plan.get('european_aqi')}

Current weather summary:
- Temperature: {plan.get('temperature')} °C
- Humidity: {plan.get('humidity')} %
- Wind speed: {plan.get('wind_speed')} km/h
- Precipitation: {plan.get('precipitation')} mm

User details:
- Profile: {plan.get('profile')}
- Activity: {plan.get('activity')}
- Duration: {plan.get('duration_minutes')} minutes


Using the provided CPCB and WHO air quality guideline context, explain:
1. Whether the user should continue, reduce, postpone, or avoid the activity.
2. Why this recommendation is suitable.
3. Which pollutant is the main concern.
4. How weather conditions may affect the recommendation if relevant.
5. A safer alternative action.

Do not give medical diagnosis or medicine advice.
Give general public air-quality guidance only.
"""

In [146]:
result=main_chain.invoke(rag_question)

In [147]:
result

'1. **Recommendation**: The user should **reduce** the activity.\n2. **Why this recommendation is suitable**: The overall AQI category is "Moderately Polluted," and the dominant pollutant is PM2.5 at 88.7 µg/m³. This level of pollution can have adverse health effects, especially for outdoor activities like jogging. Reducing the activity can help minimize exposure to PM2.5, which is known to penetrate deep into the lungs and cause respiratory issues.\n3. **Main concern**: The main concern is **PM2.5**. PM2.5 is particularly harmful as it can penetrate deep into the lungs and cause respiratory and cardiovascular problems.\n4. **Effect of weather conditions**: The high humidity (92%) and low wind speed (4.8 km/h) may contribute to the accumulation of pollutants in the air, making the air quality worse. This can further increase the risk of health impacts from PM2.5.\n5. **Safer alternative action**: The user can consider **indoor activities** or **jogging in a less polluted area** if avai

In [138]:
final_response = {
    "city": plan["city"],
    "pm2_5": plan["pm2_5"],
    "pm10": plan["pm10"],
    "overall_category": plan["overall_category"],
    "dominant_pollutant": plan["dominant_pollutant"],
    "basic_recommendation": plan["recommendation"],
    "rag_explanation": result
}

final_response

{'city': 'Bhubaneswar',
 'pm2_5': 88.7,
 'pm10': 97.8,
 'overall_category': 'Moderately Polluted',
 'dominant_pollutant': 'PM2.5',
 'basic_recommendation': 'You can go outside, but avoid long or intense outdoor exercise.',
 'rag_explanation': 'Based on the provided CPCB and WHO guidelines, the current air quality category is Moderately Polluted, with PM2.5 being the dominant pollutant at 88.7 µg/m³. This level of PM2.5 can have significant health impacts, particularly on individuals with respiratory conditions.\n\nGiven that the user is a student and plans to do outdoor jogging for 30 minutes, it is advisable to take the following actions to minimize health risks:\n\n1. **Limit Outdoor Activity Duration**: Reduce the duration of outdoor jogging to 15-20 minutes instead of 30 minutes to minimize exposure to PM2.5.\n\n2. **Choose a Less Polluted Time**: If possible, jog during the early morning or late evening when air quality is generally better.\n\n3. **Monitor Health**: Pay attention 

In [139]:
print("AIR QUALITY ACTION PLAN")
print("-" * 50)

print(f"City: {final_response['city']}")
print(f"PM2.5: {final_response['pm2_5']} µg/m³")
print(f"PM10: {final_response['pm10']} µg/m³")
print(f"Overall Category: {final_response['overall_category']}")
print(f"Dominant Pollutant: {final_response['dominant_pollutant']}")

print("\nBasic Recommendation:")
print(final_response["basic_recommendation"])

print("\nRAG-Based Explanation:")
print(final_response["rag_explanation"])

AIR QUALITY ACTION PLAN
--------------------------------------------------
City: Bhubaneswar
PM2.5: 88.7 µg/m³
PM10: 97.8 µg/m³
Overall Category: Moderately Polluted
Dominant Pollutant: PM2.5

Basic Recommendation:
You can go outside, but avoid long or intense outdoor exercise.

RAG-Based Explanation:
Based on the provided CPCB and WHO guidelines, the current air quality category is Moderately Polluted, with PM2.5 being the dominant pollutant at 88.7 µg/m³. This level of PM2.5 can have significant health impacts, particularly on individuals with respiratory conditions.

Given that the user is a student and plans to do outdoor jogging for 30 minutes, it is advisable to take the following actions to minimize health risks:

1. **Limit Outdoor Activity Duration**: Reduce the duration of outdoor jogging to 15-20 minutes instead of 30 minutes to minimize exposure to PM2.5.

2. **Choose a Less Polluted Time**: If possible, jog during the early morning or late evening when air quality is gener

## Evaluation

In [140]:
aqi_test_cases = [
    {"pm2_5": 20, "pm10": 40, "expected": "Good"},
    {"pm2_5": 45, "pm10": 80, "expected": "Satisfactory"},
    {"pm2_5": 75, "pm10": 150, "expected": "Moderately Polluted"},
    {"pm2_5": 100, "pm10": 220, "expected": "Poor"},
    {"pm2_5": 180, "pm10": 370, "expected": "Very Poor"},
    {"pm2_5": 280, "pm10": 460, "expected": "Severe"},
]

In [141]:
correct = 0

for case in aqi_test_cases:
    result = get_overall_category(case["pm2_5"], case["pm10"])
    predicted = result["overall_category"]
    expected = case["expected"]

    print("PM2.5:", case["pm2_5"], "| PM10:", case["pm10"])
    print("Expected:", expected)
    print("Predicted:", predicted)
    print("Pass:", predicted == expected)
    print("-" * 50)

    if predicted == expected:
        correct += 1

accuracy = correct / len(aqi_test_cases)

print("AQI Rule Engine Accuracy:", accuracy)

PM2.5: 20 | PM10: 40
Expected: Good
Predicted: Good
Pass: True
--------------------------------------------------
PM2.5: 45 | PM10: 80
Expected: Satisfactory
Predicted: Satisfactory
Pass: True
--------------------------------------------------
PM2.5: 75 | PM10: 150
Expected: Moderately Polluted
Predicted: Moderately Polluted
Pass: True
--------------------------------------------------
PM2.5: 100 | PM10: 220
Expected: Poor
Predicted: Poor
Pass: True
--------------------------------------------------
PM2.5: 180 | PM10: 370
Expected: Very Poor
Predicted: Very Poor
Pass: True
--------------------------------------------------
PM2.5: 280 | PM10: 460
Expected: Severe
Predicted: Severe
Pass: True
--------------------------------------------------
AQI Rule Engine Accuracy: 1.0


In [142]:
retrieval_eval_queries = [
    {
        "query": "What is PM2.5?",
        "expected_keywords": ["PM", "PM2.5", "respiratory", "health"]
    },
    {
        "query": "What are the AQI categories?",
        "expected_keywords": ["Good", "Satisfactory", "Poor", "Very Poor", "Severe"]
    },
    {
        "query": "How is AQI calculated?",
        "expected_keywords": ["Sub-indices", "pollutants", "worst", "AQI"]
    },
    {
        "query": "What should people do during very poor or severe AQI?",
        "expected_keywords": ["severe", "very poor", "minimize travel", "public transport"]
    },
    {
        "query": "What is PM10?",
        "expected_keywords": ["PM10", "respiratory", "health"]
    }
]

In [143]:
def evaluate_retriever(retriever, eval_queries, k=5):
    results = []

    for item in eval_queries:
        query = item["query"]
        expected_keywords = item["expected_keywords"]

        docs = retriever.invoke(query)
        combined_text = " ".join([doc.page_content for doc in docs]).lower()

        matched_keywords = [
            keyword for keyword in expected_keywords
            if keyword.lower() in combined_text
        ]

        score = len(matched_keywords) / len(expected_keywords)

        results.append({
            "query": query,
            "expected_keywords": expected_keywords,
            "matched_keywords": matched_keywords,
            "retrieval_score": score,
            "top_sources": [
                {
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                    "organization": doc.metadata.get("organization")
                }
                for doc in docs[:3]
            ]
        })

    return results

In [144]:
retrieval_results = evaluate_retriever(
    retriever=retriever,
    eval_queries=retrieval_eval_queries
)

for result in retrieval_results:
    print("\nQuery:", result["query"])
    print("Matched Keywords:", result["matched_keywords"])
    print("Retrieval Score:", result["retrieval_score"])
    print("Top Sources:", result["top_sources"])
    print("-" * 80)


Query: What is PM2.5?
Matched Keywords: ['PM', 'PM2.5', 'respiratory', 'health']
Retrieval Score: 1.0
Top Sources: [{'source': 'CPCB_AQI.pdf', 'page': 28, 'organization': 'CPCB'}, {'source': 'CPCB_AQI.pdf', 'page': 20, 'organization': 'CPCB'}, {'source': 'CPCB_AQI.pdf', 'page': 33, 'organization': 'CPCB'}]
--------------------------------------------------------------------------------

Query: What are the AQI categories?
Matched Keywords: ['Good', 'Satisfactory', 'Poor', 'Very Poor', 'Severe']
Retrieval Score: 1.0
Top Sources: [{'source': 'CPCB_AQI.pdf', 'page': 37, 'organization': 'CPCB'}, {'source': 'CPCB_AQI.pdf', 'page': 44, 'organization': 'CPCB'}, {'source': 'CPCB_AQI.pdf', 'page': 6, 'organization': 'CPCB'}]
--------------------------------------------------------------------------------

Query: How is AQI calculated?
Matched Keywords: ['pollutants', 'worst', 'AQI']
Retrieval Score: 0.75
Top Sources: [{'source': 'CPCB_AQI.pdf', 'page': 45, 'organization': 'CPCB'}, {'source': '